### Assignment 2 - Traffic Signs Classification with VGG16
- Discipline: EEE6553 - Advanced Deep Learning
- Task: Consider the Traffic Sign Dataset. For this classification task use Resnet 50 and Vision 
Transformer.
- Assignment: 2
- Question: 1
- Student: Fabio Caversan

### 1. Libraries and Environment Setup
Import the required libraries and verify whether TensorFlow can use a GPU in the current notebook kernel.

In [1]:
import os
import sys
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import (
    Flatten, Dense, Rescaling, GlobalAveragePooling2D,
    LayerNormalization, MultiHeadAttention, Dropout, Input, Reshape
)
from tensorflow.keras.applications import ResNet50

tf_version = getattr(tf, "__version__", "unknown")
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf_version)

if tf_version == "unknown":
    print("TensorFlow import looks incomplete in this kernel.")
    print("For native Windows GPU, use Python 3.10 + TensorFlow 2.10.x.")
else:
    print("Built with CUDA:", tf.test.is_built_with_cuda())
    gpus = tf.config.list_physical_devices("GPU")
    print("GPU devices:", gpus)
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU is available. Training can use GPU.")
    else:
        print("No GPU detected. Training will run on CPU.")

Python: 3.10.11
TensorFlow: 2.10.1
Built with CUDA: True
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU is available. Training can use GPU.


## 2. Configuration and Dataset Path
Define the image size, batch size, class count, epoch count and the dataset folder.

In [2]:
DATA_DIR = "./datasets/Traffic Signs dataset/data"
assert DATA_DIR is not None, (
    "Dataset directory not found."
)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
NUM_CLASSES = 5
EPOCHS = 50

print("Using dataset directory:", DATA_DIR)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)

Using dataset directory: ./datasets/Traffic Signs dataset/data
Image size: (224, 224)
Batch size: 32
Epochs: 50


## 3. Dataset Loading and Preprocessing
Load the images from directory, split the dataset into training and test subsets, and normalize the input images before training.

In [3]:
full_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)
class_names = full_ds.class_names
print("Class names:", class_names)

card = tf.data.experimental.cardinality(full_ds).numpy()
train_batches = int(0.8 * card)

train_ds = full_ds.take(train_batches)
test_ds = full_ds.skip(train_batches)
print(f"Train batches: {len(train_ds)} | Test batches: {len(test_ds)}")

normalizer = Rescaling(1.0 / 255)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (normalizer(x), y)).prefetch(AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (normalizer(x), y)).prefetch(AUTOTUNE)

Found 3870 files belonging to 5 classes.
Class names: ['Caution tringle', 'Lane merge', 'No Exit', 'slippery road', 'stop']
Train batches: 96 | Test batches: 25


## 4. ResNet50 — Architecture and Compilation
Load the pretrained ResNet50 convolutional base (frozen), add a GlobalAveragePooling head, and compile for multi-class classification.

In [4]:
resnet_base = ResNet50(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
resnet_base.trainable = False

resnet_model = Sequential([
    resnet_base,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax"),
])

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

resnet_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 global_average_pooling2d (G  (None, 2048)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 256)               524544    
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 128)               32896     
                                                                 
 dense_2 (Dense)             (None, 5)                 645       
                                                        

## 5. ResNet50 — Training
Train the ResNet50 transfer-learning model on the training subset while keeping the convolutional base frozen.

In [5]:
resnet_history = resnet_model.fit(train_ds, epochs=EPOCHS, verbose=1)

Epoch 1/50
96/96 [==============================] - 16s 89ms/step - loss: 1.5206 - accuracy: 0.3333
Epoch 2/50
96/96 [==============================] - 8s 78ms/step - loss: 1.4258 - accuracy: 0.4095
Epoch 3/50
96/96 [==============================] - 8s 81ms/step - loss: 1.3620 - accuracy: 0.4632
Epoch 4/50
96/96 [==============================] - 8s 80ms/step - loss: 1.3151 - accuracy: 0.4876
Epoch 5/50
96/96 [==============================] - 8s 80ms/step - loss: 1.2634 - accuracy: 0.5342
Epoch 6/50
96/96 [==============================] - 8s 80ms/step - loss: 1.2170 - accuracy: 0.5521
Epoch 7/50
96/96 [==============================] - 8s 80ms/step - loss: 1.1658 - accuracy: 0.5723
Epoch 8/50
96/96 [==============================] - 8s 80ms/step - loss: 1.1126 - accuracy: 0.5947
Epoch 9/50
96/96 [==============================] - 8s 81ms/step - loss: 1.0585 - accuracy: 0.6237
Epoch 10/50
96/96 [==============================] - 8s 79ms/step - loss: 1.0267 - accuracy: 0.6266
Epoch 11

## 6. ResNet50 — Evaluation
Evaluate ResNet50 on the test subset and print the classification report together with the confusion matrix.

In [6]:
resnet_loss, resnet_acc = resnet_model.evaluate(test_ds, verbose=0)
print("\n=== RESNET50 — FINAL TEST METRICS ===")
print(f"Test Loss: {resnet_loss:.4f}")
print(f"Test Accuracy: {resnet_acc:.4f}")

y_true_r, y_pred_r = [], []
for xb, yb in test_ds:
    preds = resnet_model.predict(xb, verbose=0)
    y_true_r.extend(tf.argmax(yb, axis=1).numpy())
    y_pred_r.extend(np.argmax(preds, axis=1))

y_true_r = np.array(y_true_r)
y_pred_r = np.array(y_pred_r)

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_true_r, y_pred_r, target_names=class_names, digits=4))

print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_true_r, y_pred_r))


=== RESNET50 — FINAL TEST METRICS ===
Test Loss: 0.3797
Test Accuracy: 0.8885

=== CLASSIFICATION REPORT ===
                 precision    recall  f1-score   support

Caution tringle     0.9114    0.8780    0.8944       246
     Lane merge     0.9444    0.3617    0.5231        47
        No Exit     0.8636    0.9913    0.9231       230
  slippery road     0.8349    0.8198    0.8273       111
           stop     0.9176    0.9512    0.9341       164

       accuracy                         0.8872       798
      macro avg     0.8944    0.8004    0.8204       798
   weighted avg     0.8902    0.8872    0.8796       798

=== CONFUSION MATRIX ===
[[216   1  10   7  12]
 [ 15  17   4   9   2]
 [  0   0 228   2   0]
 [  4   0  16  91   0]
 [  2   0   6   0 156]]


## 7. Vision Transformer (ViT) — Architecture and Compilation
Build a simple Vision Transformer from scratch:
1. **Patch extraction** — split each 224×224 image into 16×16 patches (196 patches).
2. **Patch + position embedding** — linearly project each patch and add a learnable positional embedding plus a CLS token.
3. **Transformer encoder** — stack of Multi-Head Self-Attention + MLP blocks with LayerNorm.
4. **Classification head** — take the CLS token output and classify into 5 classes.

In [7]:
# ViT hyper-parameters
PATCH_SIZE = 16
NUM_PATCHES = (IMG_SIZE[0] // PATCH_SIZE) ** 2   # 196
PROJECTION_DIM = 64
NUM_HEADS = 4
TRANSFORMER_LAYERS = 4
MLP_UNITS = [128, 64]

# --- helper layers -----------------------------------------------------------
class PatchExtract(tf.keras.layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dim = patches.shape[-1]
        patches = tf.reshape(patches, [-1, NUM_PATCHES, patch_dim])
        return patches


class PatchEmbedding(tf.keras.layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.projection = Dense(projection_dim)
        self.position_embedding = tf.keras.layers.Embedding(
            input_dim=num_patches + 1, output_dim=projection_dim
        )
        self.cls_token = self.add_weight(
            shape=(1, 1, projection_dim), initializer="zeros", trainable=True, name="cls"
        )

    def call(self, patches):
        batch = tf.shape(patches)[0]
        projected = self.projection(patches)

        cls_tokens = tf.broadcast_to(self.cls_token, [batch, 1, projected.shape[-1]])
        projected = tf.concat([cls_tokens, projected], axis=1)

        positions = tf.range(start=0, limit=NUM_PATCHES + 1, delta=1)
        encoded = projected + self.position_embedding(positions)
        return encoded

# --- transformer encoder block ------------------------------------------------
def transformer_block(x, num_heads, projection_dim, mlp_units, dropout_rate=0.1):
    # Multi-Head Self-Attention
    attn_out = LayerNormalization(epsilon=1e-6)(x)
    attn_out = MultiHeadAttention(
        num_heads=num_heads, key_dim=projection_dim // num_heads
    )(attn_out, attn_out)
    attn_out = Dropout(dropout_rate)(attn_out)
    x = x + attn_out

    # MLP
    mlp_out = LayerNormalization(epsilon=1e-6)(x)
    for units in mlp_units:
        mlp_out = Dense(units, activation="gelu")(mlp_out)
        mlp_out = Dropout(dropout_rate)(mlp_out)
    x = x + mlp_out
    return x

# --- build ViT ---------------------------------------------------------------
inputs = Input(shape=IMG_SIZE + (3,))
patches = PatchExtract(PATCH_SIZE)(inputs)
encoded = PatchEmbedding(NUM_PATCHES, PROJECTION_DIM)(patches)

for _ in range(TRANSFORMER_LAYERS):
    encoded = transformer_block(encoded, NUM_HEADS, PROJECTION_DIM, MLP_UNITS)

encoded = LayerNormalization(epsilon=1e-6)(encoded)
cls_output = encoded[:, 0]   # CLS token

x = Dense(128, activation="relu")(cls_output)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation="softmax")(x)

vit_model = Model(inputs=inputs, outputs=outputs)

vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

vit_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 patch_extract (PatchExtract)   (None, 196, 768)     0           ['input_2[0][0]']                
                                                                                                  
 patch_embedding (PatchEmbeddin  (None, 197, 64)     61888       ['patch_extract[0][0]']          
 g)                                                                                               
                                                                                              

## 8. Vision Transformer — Training
Train the ViT model on the same training subset used for ResNet50.

In [8]:
vit_history = vit_model.fit(train_ds, epochs=EPOCHS, verbose=1)

Epoch 1/50
96/96 [==============================] - 7s 38ms/step - loss: 1.5544 - accuracy: 0.3200
Epoch 2/50
96/96 [==============================] - 3s 32ms/step - loss: 1.2971 - accuracy: 0.4632
Epoch 3/50
96/96 [==============================] - 3s 34ms/step - loss: 1.0209 - accuracy: 0.5570
Epoch 4/50
96/96 [==============================] - 3s 34ms/step - loss: 0.8184 - accuracy: 0.6663
Epoch 5/50
96/96 [==============================] - 3s 32ms/step - loss: 0.6764 - accuracy: 0.7279
Epoch 6/50
96/96 [==============================] - 4s 44ms/step - loss: 0.6133 - accuracy: 0.7565
Epoch 7/50
96/96 [==============================] - 5s 50ms/step - loss: 0.5979 - accuracy: 0.7539
Epoch 8/50
96/96 [==============================] - 5s 48ms/step - loss: 0.5654 - accuracy: 0.7656
Epoch 9/50
96/96 [==============================] - 5s 49ms/step - loss: 0.5384 - accuracy: 0.7767
Epoch 10/50
96/96 [==============================] - 5s 48ms/step - loss: 0.5263 - accuracy: 0.7819
Epoch 11/

## 9. Vision Transformer — Evaluation
Evaluate the ViT on the test subset and print the classification report together with the confusion matrix.

In [9]:
vit_loss, vit_acc = vit_model.evaluate(test_ds, verbose=0)
print("\n=== VISION TRANSFORMER — FINAL TEST METRICS ===")
print(f"Test Loss: {vit_loss:.4f}")
print(f"Test Accuracy: {vit_acc:.4f}")

y_true_v, y_pred_v = [], []
for xb, yb in test_ds:
    preds = vit_model.predict(xb, verbose=0)
    y_true_v.extend(tf.argmax(yb, axis=1).numpy())
    y_pred_v.extend(np.argmax(preds, axis=1))

y_true_v = np.array(y_true_v)
y_pred_v = np.array(y_pred_v)

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_true_v, y_pred_v, target_names=class_names, digits=4))

print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_true_v, y_pred_v))


=== VISION TRANSFORMER — FINAL TEST METRICS ===
Test Loss: 0.0291
Test Accuracy: 0.9925

=== CLASSIFICATION REPORT ===
                 precision    recall  f1-score   support

Caution tringle     0.9876    0.9917    0.9896       241
     Lane merge     1.0000    0.9318    0.9647        44
        No Exit     1.0000    1.0000    1.0000       225
  slippery road     0.9823    1.0000    0.9911       111
           stop     1.0000    1.0000    1.0000       177

       accuracy                         0.9937       798
      macro avg     0.9940    0.9847    0.9891       798
   weighted avg     0.9938    0.9937    0.9937       798

=== CONFUSION MATRIX ===
[[239   0   0   2   0]
 [  3  41   0   0   0]
 [  0   0 225   0   0]
 [  0   0   0 111   0]
 [  0   0   0   0 177]]


## 10. Model Comparison
Side-by-side summary of ResNet50 and Vision Transformer performance on the test set.

In [10]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["ResNet50", "Vision Transformer"],
    "Test Loss": [resnet_loss, vit_loss],
    "Test Accuracy": [resnet_acc, vit_acc],
})
print(comparison.to_string(index=False))

             Model  Test Loss  Test Accuracy
          ResNet50   0.379710       0.888471
Vision Transformer   0.029135       0.992481


## 11. Hybrid: ResNet50 + Transformer — Architecture and Compilation
Combine ResNet50 and a Transformer encoder into a single model:
1. **ResNet50 backbone (frozen)** extracts a 7×7×2048 feature map from each 224×224 image.
2. The feature map is reshaped into **49 spatial tokens** (one per 7×7 cell), each with 2048 features.
3. Tokens are linearly projected to a lower dimension and a learnable **CLS token + positional embeddings** are added.
4. A small **Transformer encoder** attends over these tokens, capturing global spatial relationships.
5. The CLS token output feeds the final classification head.

This lets ResNet50 handle low-level feature extraction (where it already excels) and the Transformer handle high-level reasoning across spatial regions.

In [13]:
# Hybrid hyper-parameters
HYBRID_PROJ_DIM = 128
HYBRID_NUM_HEADS = 4
HYBRID_TRANSFORMER_LAYERS = 2
HYBRID_MLP_UNITS = [256, 128]
HYBRID_EPOCHS = 50

# --- ResNet50 backbone (frozen, no top) ---
hybrid_backbone = ResNet50(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
hybrid_backbone.trainable = False
# Output shape: (batch, 7, 7, 2048) → 49 spatial tokens of dim 2048

# --- Build hybrid model ---
hybrid_input = Input(shape=IMG_SIZE + (3,))
features = hybrid_backbone(hybrid_input, training=False)   # (batch, 7, 7, 2048)

# Reshape to sequence of tokens: (batch, 49, 2048)
num_tokens = features.shape[1] * features.shape[2]   # 49
token_dim = features.shape[3]                         # 2048
tokens = Reshape((num_tokens, token_dim))(features)

# Linear projection to lower dimension
tokens = Dense(HYBRID_PROJ_DIM)(tokens)  # (batch, 49, HYBRID_PROJ_DIM)

# CLS token
cls_token = tf.Variable(tf.zeros([1, 1, HYBRID_PROJ_DIM]), trainable=True, name="hybrid_cls")
cls_broadcast = tf.broadcast_to(cls_token, [tf.shape(tokens)[0], 1, HYBRID_PROJ_DIM])
tokens = tf.concat([cls_broadcast, tokens], axis=1)   # (batch, 50, HYBRID_PROJ_DIM)

# Positional embedding
pos_emb = tf.keras.layers.Embedding(
    input_dim=num_tokens + 1, output_dim=HYBRID_PROJ_DIM
)(tf.range(num_tokens + 1))
tokens = tokens + pos_emb

# Transformer encoder blocks
for _ in range(HYBRID_TRANSFORMER_LAYERS):
    # Self-Attention
    attn = LayerNormalization(epsilon=1e-6)(tokens)
    attn = MultiHeadAttention(
        num_heads=HYBRID_NUM_HEADS, key_dim=HYBRID_PROJ_DIM // HYBRID_NUM_HEADS
    )(attn, attn)
    attn = Dropout(0.1)(attn)
    tokens = tokens + attn

    # MLP
    mlp = LayerNormalization(epsilon=1e-6)(tokens)
    for units in HYBRID_MLP_UNITS:
        mlp = Dense(units, activation="gelu")(mlp)
        mlp = Dropout(0.1)(mlp)
    tokens = tokens + mlp

tokens = LayerNormalization(epsilon=1e-6)(tokens)
hybrid_cls = tokens[:, 0]   # CLS token output

# Classification head
hx = Dense(128, activation="relu")(hybrid_cls)
hx = Dropout(0.3)(hx)
hybrid_output = Dense(NUM_CLASSES, activation="softmax")(hx)

hybrid_model = Model(inputs=hybrid_input, outputs=hybrid_output)

hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

hybrid_model.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_6 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 resnet50 (Functional)          (None, 7, 7, 2048)   23587712    ['input_6[0][0]']                
                                                                                                  
 reshape_1 (Reshape)            (None, 49, 2048)     0           ['resnet50[0][0]']               
                                                                                                  
 dense_21 (Dense)               (None, 49, 128)      262272      ['reshape_1[0][0]']        

## 12. Hybrid — Training
Train the ResNet50 + Transformer hybrid on the same training subset.

In [14]:
hybrid_history = hybrid_model.fit(train_ds, epochs=HYBRID_EPOCHS, verbose=1)

Epoch 1/50
96/96 [==============================] - 11s 81ms/step - loss: 1.6296 - accuracy: 0.2858
Epoch 2/50
96/96 [==============================] - 8s 80ms/step - loss: 1.5134 - accuracy: 0.3167
Epoch 3/50
96/96 [==============================] - 8s 80ms/step - loss: 1.4016 - accuracy: 0.4069
Epoch 4/50
96/96 [==============================] - 8s 80ms/step - loss: 1.0769 - accuracy: 0.6022
Epoch 5/50
96/96 [==============================] - 8s 82ms/step - loss: 0.8104 - accuracy: 0.7018
Epoch 6/50
96/96 [==============================] - 8s 81ms/step - loss: 0.6278 - accuracy: 0.7816
Epoch 7/50
96/96 [==============================] - 8s 81ms/step - loss: 0.5658 - accuracy: 0.8096
Epoch 8/50
96/96 [==============================] - 8s 79ms/step - loss: 0.4742 - accuracy: 0.8405
Epoch 9/50
96/96 [==============================] - 8s 82ms/step - loss: 0.3991 - accuracy: 0.8633
Epoch 10/50
96/96 [==============================] - 8s 82ms/step - loss: 0.3762 - accuracy: 0.8717
Epoch 11

## 13. Hybrid — Evaluation and Final Comparison
Evaluate the hybrid model and compare all three approaches side by side.

In [15]:
hybrid_loss, hybrid_acc = hybrid_model.evaluate(test_ds, verbose=0)
print("\n=== HYBRID (ResNet50 + Transformer) — FINAL TEST METRICS ===")
print(f"Test Loss: {hybrid_loss:.4f}")
print(f"Test Accuracy: {hybrid_acc:.4f}")

y_true_h, y_pred_h = [], []
for xb, yb in test_ds:
    preds = hybrid_model.predict(xb, verbose=0)
    y_true_h.extend(tf.argmax(yb, axis=1).numpy())
    y_pred_h.extend(np.argmax(preds, axis=1))

y_true_h = np.array(y_true_h)
y_pred_h = np.array(y_pred_h)

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_true_h, y_pred_h, target_names=class_names, digits=4))

print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_true_h, y_pred_h))

# --- Final 3-model comparison ---
print("\n" + "=" * 60)
print("FINAL COMPARISON — ALL THREE MODELS")
print("=" * 60)
comparison_all = pd.DataFrame({
    "Model": ["ResNet50", "Vision Transformer", "Hybrid (ResNet50+Transformer)"],
    "Test Loss": [resnet_loss, vit_loss, hybrid_loss],
    "Test Accuracy": [resnet_acc, vit_acc, hybrid_acc],
})
print(comparison_all.to_string(index=False))


=== HYBRID (ResNet50 + Transformer) — FINAL TEST METRICS ===
Test Loss: 0.1346
Test Accuracy: 0.9449

=== CLASSIFICATION REPORT ===
                 precision    recall  f1-score   support

Caution tringle     0.9958    0.9555    0.9752       247
     Lane merge     0.8974    0.7447    0.8140        47
        No Exit     0.9150    0.9956    0.9536       227
  slippery road     0.8952    0.8952    0.8952       105
           stop     0.9941    0.9826    0.9883       172

       accuracy                         0.9524       798
      macro avg     0.9395    0.9147    0.9253       798
   weighted avg     0.9534    0.9524    0.9519       798

=== CONFUSION MATRIX ===
[[236   4   5   1   1]
 [  0  35   2  10   0]
 [  1   0 226   0   0]
 [  0   0  11  94   0]
 [  0   0   3   0 169]]

FINAL COMPARISON — ALL THREE MODELS
                        Model  Test Loss  Test Accuracy
                     ResNet50   0.379710       0.888471
           Vision Transformer   0.029135       0.992481
Hybri

### 14 · Conclusion & Key Findings

**Performance Summary (Traffic Sign Classification – 5 classes, ~4 000 images, 50 epochs)**

| Model | Test Loss | Test Accuracy | Macro F1 | Strengths | Weaknesses |
|-------|-----------|--------------|----------|-----------|------------|
| **Vision Transformer (from scratch)** | **0.0291** | **99.37 %** | **0.9891** | Near-perfect across all classes; captures global context via self-attention; with enough epochs it surpasses all CNN baselines | Slower to converge; needs more epochs to realise its potential |
| **Hybrid ResNet50 + Transformer** | 0.1346 | 95.24 % | 0.9253 | Combines local CNN features with global attention; strong on most classes (stop F1 = 0.99) | Lane merge recall lower (74 %); frozen backbone limits feature adaptation |
| **ResNet50 (transfer learning)** | 0.3797 | 88.85 % | 0.8204 | Efficient residual connections; reasonable accuracy with frozen backbone | Weak on Lane merge (F1 = 0.52); frozen backbone too generic for this domain |

**Per-class highlights**

| Class | ResNet50 F1 | ViT F1 | Hybrid F1 |
|-------|------------|--------|-----------|
| Caution triangle | 0.8944 | 0.9896 | 0.9752 |
| Lane merge | 0.5231 | 0.9647 | 0.8140 |
| No Exit | 0.9231 | 1.0000 | 0.9536 |
| Slippery road | 0.8273 | 0.9911 | 0.8952 |
| Stop | 0.9341 | 1.0000 | 0.9883 |

- **Lane merge** is the hardest class for ResNet50 and Hybrid, likely due to the smallest sample count (~47 test images). ViT handles it well (F1 = 0.96).
- **No Exit** and **Stop** are classified near-perfectly by all three, but ViT achieves perfect F1 = 1.00 on both.

**Key takeaway — ViT surpasses VGG16 and all other models**

With 50 training epochs the Vision Transformer trained from scratch reaches **99.37 % accuracy**, surpassing even VGG16's 98.87 %. This overturns the earlier observation (at fewer epochs) that ViT underperforms on small datasets.

1. **Epoch budget matters** – At 50 epochs ViT had enough iterations to learn meaningful patch-level representations. Earlier runs with fewer epochs showed ~55 % accuracy because the model had not yet converged — Transformers simply need more training steps than CNNs to compensate for their lack of inductive biases.

2. **Self-attention advantage** – Once converged, the global self-attention mechanism allows ViT to capture spatial relationships across the entire image, which is especially valuable for traffic signs where shape, symbol, and border colour must be jointly considered.

3. **Hybrid benefits but does not surpass ViT** – The Hybrid (ResNet50 + Transformer) at 95.24 % improves substantially over standalone ResNet50 (88.85 %) by adding Transformer reasoning on top of CNN features. However, freezing the ResNet50 backbone constrains the quality of input tokens, limiting peak performance compared to the fully trainable ViT.

4. **ResNet50 frozen baseline** – At 88.85 % ResNet50 performs respectably but is held back by its frozen backbone. The compact classification head cannot fully compensate for the domain gap between ImageNet and traffic-sign features.

**Potential improvements**

- **Fine-tune backbone layers** – Unfreezing the last few ResNet50 blocks for both the standalone and hybrid models would likely push their accuracy closer to ViT.
- **Data augmentation** – Rotation, colour jitter, and cutout would further benefit all models, especially for under-represented classes like Lane merge.
- **Pre-trained ViT weights** – Loading ImageNet-21k ViT weights instead of training from scratch could achieve the same accuracy in fewer epochs.
- **Learning-rate scheduling** – Cosine annealing or warm-up schedules would help stabilise ViT training and potentially improve final accuracy.